# Funciones

In [ ]:
# Librerías estándar
import os
import warnings

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización de datos
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Análisis de nulos
import missingno as msno

# Estadística
import scipy.stats as stats

# Configuración de warnings
warnings.filterwarnings('ignore')

## Cargar de datos

In [ ]:
def leer_archivo(ruta_completa):
    try:

        _, extension = os.path.splitext(ruta_completa.lower())


        if extension == '.csv':
            df = pd.read_csv(ruta_completa)
        elif extension in ('.xlsx', '.xls'):
            df = pd.read_excel(ruta_completa)
        else:
            print("Error: Formato no compatible")
            return None

        return df

    except FileNotFoundError:
        print(f"Error: Archivo no encontrado en la ruta '{ruta_completa}'.")
        return None

    except Exception as e:
        print(f"Error inesperado: {e}")
        return None


### exploracion_inicial

In [ ]:
def exploracion_inicial(df, nombre=None, tipo=None):
    """
    Realiza una exploración inicial de un DataFrame y muestra información clave.

    Parámetros:
    df (pd.DataFrame): El DataFrame a explorar.
    tipo (str, opcional): El tipo de exploración. 'simple' muestra menos detalles.

    Imprime:
    Información relevante sobre el DataFrame, incluyendo filas, columnas, tipos de datos,
    estadísticas descriptivas, y valores nulos.
    """
    if nombre:
      print(nombre.upper().center(90, ' # '))
      print('\n\n')

    # Información básica sobre el DataFrame
    num_filas, num_columnas = df.shape
    print(f"¿Cuántas filas y columnas hay en el conjunto de datos?")
    print(f"\tHay {num_filas:,} filas y {num_columnas:,} columnas.")
    print('#' * 90)

    # Exploración simple
    if tipo == 'simple':
        print("¿Cuáles son las primeras dos filas del conjunto de datos?")
        display(df.head(2))
    else:
        # Exploración completa
        print("¿Cuáles son las primeras cinco filas del conjunto de datos?")
        display(df.head())
        print('-' * 100)

        print("¿Cuáles son las últimas cinco filas del conjunto de datos?")
        display(df.tail())
        print('-' * 100)

        print("¿Cómo puedes obtener una muestra aleatoria de filas del conjunto de datos?")
        display(df.sample(n=5))
        print('-' * 100)

        print("¿Cuáles son las columnas del conjunto de datos?")
        print("\n".join(f"\t- {col}" for col in df.columns))
        print('-' * 100)

        print("¿Cuál es el tipo de datos de cada columna?")
        print(df.dtypes)
        print('-' * 100)

        print("¿Cuántas columnas hay de cada tipo de datos?")
        print(df.dtypes.value_counts())
        print('-' * 100)

        print("¿Cómo podríamos obtener información más completa sobre la estructura y el contenido del DataFrame?")
        print(df.info())
        print('-' * 100)

        print("¿Cuántos valores únicos tiene cada columna?")
        print(df.nunique())
        print('-' * 100)

        print("¿Cuáles son los valores únicos de cada columna?")
        df_valores_unicos = pd.DataFrame(df.apply(lambda x: x.unique()))
        display(df_valores_unicos)
        print('-' * 100)

        print("¿Cuáles son las estadísticas descriptivas básicas de todas las columnas?")
        display(df.describe(include='all').fillna(''))
        print('-' * 100)

        print("¿Cuántos valores nulos hay en cada columna del DataFrame?")
        display(df.isnull().sum())
        print('-' * 100)

        print("¿Cuál es el porcentaje de valores nulos por columna, ordenado de mayor a menor?")
        df_nulos = df.isnull().sum().div(len(df)).mul(100).round(2).reset_index().rename(columns = {'index': 'Col', 0: 'pct'})
        df_nulos = df_nulos.sort_values(by = 'pct', ascending=False).reset_index(drop = True)
        display(df_nulos)
        print('-' * 100)

        print("## Valores nulos: Visualización")
        msno.bar(df, figsize = (6, 3), fontsize= 9)
        plt.show()
        print('-' * 100)

        print("## Visualización de patrones en valores nulos")
        msno.matrix(df, figsize = (6, 3), fontsize= 9, sparkline = False)
        plt.show()
        print('-' * 100)

        print("¿Existen relaciones entre los valores faltantes de distintas columnas?")
        msno.heatmap(df, figsize = (6, 3), fontsize= 9)
        plt.show()
        print('-' * 100)

    print('#' * 90)

### Deteccion_de_outliers

In [ ]:
def deteccion_outliers (df, variable):
    columna = df[variable]

    sns.boxplot(
      data=df,
      y=variable,
    )
    plt.show()

    Q1 = columna.quantile(0.25)
    Q3 = columna.quantile(0.75)
    IQR = Q3 - Q1

    print('Valor del segundo cuartil (25%): {:.2f}'.format(Q1))
    print('Valor del tercer cuartil (75%): {:.2f}'.format(Q3))
    print('Valor del rango intercuartil (IQR): {:.2f}'.format(IQR))

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    print(f"Los valores atípicos se definen como aquellos que caen fuera del siguiente rango:")
    print(f"\t - Límite inferior (considerado extremadamente bajo): {limite_inferior:.2f}")
    print(f"\t - Límite superior (considerado extremadamente alto): {limite_superior:.2f}")

    outliers = list(columna[((columna < limite_inferior) | (columna > limite_superior))].index)
    num_outliers = len(outliers)
    print(f"Hay {num_outliers} outliers en la variable '{variable}'")
    return outliers

### Deteccion_de_outliers_multivariable

In [ ]:
def deteccion_outliers_multivariable (df, variable, group_value):
        for g, grupo in df.groupby(group_value):
            columna = grupo[variable]

            sns.boxplot(data=grupo, y=variable)
            plt.title(f"Boxplot - {group_value} = {g}")
            plt.show()

            Q1 = columna.quantile(0.25)
            Q3 = columna.quantile(0.75)
            IQR = Q3 - Q1

            print(f"\n[{group_value} = {g}]")
            print('Valor del segundo cuartil (25%): {:.2f}'.format(Q1))
            print('Valor del tercer cuartil (75%): {:.2f}'.format(Q3))
            print('Valor del rango intercuartil (IQR): {:.2f}'.format(IQR))

            limite_inferior = Q1 - 1.5 * IQR
            limite_superior = Q3 + 1.5 * IQR

            print(f"Los valores atípicos se definen como aquellos que caen fuera del siguiente rango:")
            print(f"\t - Límite inferior (considerado extremadamente bajo): {limite_inferior:.2f}")
            print(f"\t - Límite superior (considerado extremadamente alto): {limite_superior:.2f}")

            outliers = list(columna[((columna < limite_inferior) | (columna > limite_superior))].index)
            num_outliers = len(outliers)
            print(f"Hay {num_outliers} outliers en la variable '{variable}' para {group_value}={g}")
    

### Graficar boxplots

In [ ]:
def graficar_boxplot_px(df, variable_analisis):
    # Crear el boxplot usando Plotly Express
    fig = px.box(df, y=variable_analisis)

    # Actualizar títulos del gráfico
    fig.update_layout(title=f'Boxplot: {variable_analisis}',
                      yaxis_title='Frecuencia',
        width=600,     # ancho en píxeles
        height=400     # alto en píxeles
                      )

    # Actualizar el fondo del gráfico a blanco
    fig.update_layout({
        'plot_bgcolor': 'rgba(255, 255, 255, 1)',
        'xaxis': {'showgrid': True, 'gridcolor': 'lightgrey'},
        'yaxis': {'showgrid': True, 'gridcolor': 'lightgrey'}
    })

    # Mostrar el gráfico
    fig.show()

### Visualización_solo_de_nulos

In [ ]:
def nulos(df, nombre=None):
    """
    Visualización de los nulos en un dataframe.
    """
    if nombre:
        print(nombre.upper().center(90, ' # '))
        print('\n\n')

    print("¿Cuántos valores nulos hay en cada columna del DataFrame?")
    display(df.isnull().sum())
    print('-' * 100)

    print("¿Cuál es el porcentaje de valores nulos por columna, ordenado de mayor a menor?")
    df_nulos = (df.isnull().sum().div(len(df)).mul(100).round(2).reset_index().rename(columns={'index': 'Col', 0: 'pct'}))
    df_nulos = df_nulos.sort_values(by='pct', ascending=False).reset_index(drop=True)
    display(df_nulos)
    print('-' * 100)

    print("## Visualización de valores nulos")
    msno.bar(df, figsize=(6, 3), fontsize=9)
    plt.show()
    print('-' * 100)

    print("## Visualización de patrones en valores nulos")
    msno.matrix(df, figsize=(6, 3), fontsize=9, sparkline=False)
    plt.show()
    print('-' * 100)

    msno.heatmap(df, figsize=(6, 3), fontsize=9)
    plt.show()
    print('#' * 90)